# Kaggle Lyric Embeddings

Generates multilingual sentence embeddings for the cleaned lyrics corpus and saves them in a format compatible with `music_rec/embeddings.py`.

## 1. Install / import dependencies

In [ ]:
import sys, subprocess

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-U', 'sentence-transformers'],
    check=False,
)

import json
import numpy as np
import pandas as pd
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__, '| Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Config

Matches `music_rec/config.py` and `music_rec/embeddings.py`.

In [ ]:
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
BATCH_SIZE = 64
LOG_EVERY = 500

## 3. Data-loading helper (checks `/kaggle/input` first, then local)

In [ ]:
from pathlib import Path


def resolve_data_file(*candidates):
    """Return the first existing match from the provided candidates."""
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path

    kaggle_root = Path('/kaggle/input')
    if kaggle_root.exists():
        for candidate in candidates:
            matches = sorted(kaggle_root.rglob(Path(candidate).name))
            if matches:
                return matches[0]

    raise FileNotFoundError(f'Could not locate any of: {candidates}')


WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
print('Working dir for artifacts:', WORK_DIR.resolve())

## 4. Load cleaned lyrics (or fall back to the raw final CSV)

In [ ]:
lyrics_path = resolve_data_file(
    'cleaned_lyrics.csv',
    '../music_rec_artifacts/cleaned_lyrics.csv',
    'music_rec_artifacts/cleaned_lyrics.csv',
    'Lyrics_Dataset_final.csv',
)
print('Using lyrics file:', lyrics_path)
df = pd.read_csv(lyrics_path, encoding='utf-8')

if 'lyrics' in df.columns:
    text_series = df['lyrics']
elif 'lyrics_devanagari' in df.columns:
    text_series = df['lyrics_devanagari']
else:
    raise KeyError("No lyrics column found (expected 'lyrics' or 'lyrics_devanagari').")

if 'song_id' in df.columns:
    song_ids = df['song_id'].tolist()
else:
    song_ids = list(range(len(df)))

texts = text_series.fillna('').astype(str).tolist()
print('Songs to embed:', len(texts))

## 5. Encode with mean pooling on GPU

Raw float32 vectors are saved and normalized later during indexing.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

embeddings = []
for start in range(0, len(texts), BATCH_SIZE):
    batch = texts[start:start + BATCH_SIZE]
    vecs = model.encode(
        batch,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )
    embeddings.append(vecs.astype(np.float32))
    done = min(start + BATCH_SIZE, len(texts))
    if done % LOG_EVERY < BATCH_SIZE or done == len(texts):
        print(f'[embeddings] {done}/{len(texts)} songs encoded')

matrix = np.vstack(embeddings).astype(np.float32)
print('Embedding matrix shape:', matrix.shape, '| dtype:', matrix.dtype)

## 6. Save `embeddings.npy` + `embedding_ids.json`

In [ ]:
emb_path = WORK_DIR / 'embeddings.npy'
ids_path = WORK_DIR / 'embedding_ids.json'

np.save(emb_path, matrix)
ids_path.write_text(json.dumps(song_ids), encoding='utf-8')

print('Saved', matrix.shape, '->', emb_path.resolve())
print('Saved', len(song_ids), 'ids ->', ids_path.resolve())
print('\nArtifacts to download from /kaggle/working:')
print('  - embeddings.npy      -> music_rec_artifacts/embeddings.npy')
print('  - embedding_ids.json  -> music_rec_artifacts/embedding_ids.json')